# 04 · Kenya → Ghana transfer (step 10) and fairness (step 11)

**Part 1, transfer (step 10):** Alice. **Decision record:** [ADR 0019](../docs/adr/0019-kenya-to-ghana-transfer.md). **Plain write-up:** [learnings/10](../learnings/10_transfer.md).
**Part 2, fairness (step 11):** Joseph (took over from Liza), below Part 1.

**The question.** NkwaPay's labelled loans are all Kenyan. Its Ghanaian applicants have **no repayment outcomes**. How different is Ghana, can the Kenyan models be adjusted for it, and what should NkwaPay see before it trusts them there?

**What we can and can't measure.**
- **Default model (Task 2):** Ghana has no labels, so we can only compare *who applies* and *how the model scores them*, and use Kenyan stand-ins.
- **Amount model (Task 1):** `Test.csv` contains the real amount of every Ghanaian loan, so here we **can** score the Kenyan model on Ghana directly.

**Privacy:** every output is a count, a share, a summary or a score. No applicant's row is printed.

In [1]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib, matplotlib.ticker
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
from src.features import build, FEATURES, preprocessor, splits
from src.amounts import usd_rate, AMOUNT_CLUES, regressor
from src.classification_experiments import final_model, logit

TABLES, FIGURES = ROOT / "learnings" / "tables", ROOT / "learnings" / "figures"
DATA = ROOT / "data" / "zindi"
tr, te = build(str(DATA))
gh, ke_test = te[te.country_id == "Ghana"].copy(), te[te.country_id == "Kenya"].copy()
for d in (tr, gh, ke_test):
    d["usd_rate"] = usd_rate(d, DATA)
CUTOFF = {"Kenya": 0.021, "Ghana": 0.032}   # ADR 0007, on calibrated probabilities only
print(len(tr), "labelled Kenyan loans |", len(ke_test), "Kenyan test loans |", len(gh), "Ghanaian test loans")

68464 labelled Kenyan loans | 15069 Kenyan test loans | 3525 Ghanaian test loans


## 1. How different are Ghana's applicants?

We put three groups side by side:
- **Kenya, learned from:** the labelled loans (`Train.csv`);
- **Kenya, test:** Kenyan applicants in `Test.csv`. This is a control: if the method works, it should find these close to the training loans;
- **Ghana:** Ghanaian applicants in `Test.csv`.

Amounts are shown in local currency and in US dollars (that year's average rate).

In [2]:
def profile(d):
    usd = d.Total_Amount / d.usd_rate
    seen_type, seen_lender = d.loan_type.isin(set(tr.loan_type)), d.lender_id.isin(set(tr.lender_id))
    return pd.Series({
        "loans": len(d), "customers": d.customer_id.nunique(),
        "also a customer in Kenya's labelled loans": d.customer_id.isin(set(tr.customer_id)).mean(),
        "most common loan type": f"{d.loan_type.value_counts().index[0]} ({d.loan_type.value_counts(normalize=True).iloc[0]:.0%})",
        "loan type never seen in Kenya's labelled loans": 1 - seen_type.mean(),
        "lender never seen in Kenya's labelled loans": 1 - seen_lender.mean(),
        "7-day loans": (d.duration == 7).mean(), "loans longer than 90 days": (d.duration > 90).mean(),
        "flagged 'New Loan'": (d.New_versus_Repeat == "New Loan").mean(),
        "first loan in the data (no earlier loans)": (d.n_prev_loans == 0).mean(),
        "earlier loans, but no known outcome": ((d.n_prev_loans > 0) & (d.no_known_history == 1)).mean(),
        "at least one known earlier outcome": (d.no_known_history == 0).mean(),
        "earlier loans (median)": d.n_prev_loans.median(),
        "amount, local currency (median)": d.Total_Amount.median(),
        "amount, US$ (median)": usd.median(), "amount, US$ (25%-75%)": f"{usd.quantile(.25):,.0f}-{usd.quantile(.75):,.0f}",
        "last earlier amount, US$ (median)": (d.last_amount / d.usd_rate).median(),
        "dates": f"{d.disbursement_date.min():%b %Y}-{d.disbursement_date.max():%b %Y}",
    })
prof = pd.DataFrame({"Kenya, learned from": profile(tr), "Kenya, test": profile(ke_test), "Ghana": profile(gh)})
prof.to_csv(TABLES / "10_profile.csv")
prof

,"Kenya, learned from","Kenya, test",Ghana
loans,68464,15069,3525
customers,6508,4516,446
also a customer in Kenya's labelled loans,1.0,0.983874,0.0
most common loan type,Type_1 (90%),Type_1 (90%),Type_3 (86%)
loan type never seen in Kenya's labelled loans,0.0,0.000199,0.872624
lender never seen in Kenya's labelled loans,0.0,0.0,0.967092
7-day loans,0.9466,0.950693,0.862128
loans longer than 90 days,0.002162,0.001725,0.135887
flagged 'New Loan',0.008267,0.006902,0.01844
first loan in the data (no earlier loans),0.092939,0.090981,0.187801


**Loan types, in detail.** For each type: its share of Ghanaian loans, how many Kenyan labelled loans it has, and the Kenyan default rate. A type with few Kenyan loans gives the model very little to learn from.

In [3]:
kt = tr.groupby("loan_type").agg(kenya_loans=("target", "size"), kenya_default_rate=("target", "mean"))
types = (gh.loan_type.value_counts(normalize=True).rename("share of Ghana").to_frame()
         .join(gh.groupby("loan_type").Total_Amount.median().div(gh.usd_rate.median()).rename("Ghana median US$"))
         .join(kt).fillna({"kenya_loans": 0}))
types.to_csv(TABLES / "10_loan_types.csv")
types.round(3)

,share of Ghana,Ghana median US$,kenya_loans,kenya_default_rate
loan_type,,,,
Type_3,0.862,295.629,0.0,NaN
Type_2,0.127,9304.463,74.0,0.608
Type_8,0.010,6220.473,0.0,NaN


**How far has each clue moved?** The **population stability index (PSI)** is the standard drift score in credit scoring. It splits a clue into bins (one per value for types and flags; for numbers, the tenths of both countries), then compares each bin's share in Kenya and in Ghana. The usual reading:
- below 0.1: stable;
- 0.1 to 0.25: some shift;
- above 0.25: a big shift, so don't trust the model on it without checking.

The money clues are scored twice: raw (cedis vs shillings) and in dollars.

In [4]:
def psi(k, g, categorical=False, eps=1e-4):
    if categorical or k.nunique() <= 20:   # few distinct values (flags, counts like n_defaults): one bin per value
        cats = sorted(set(k.astype(str)) | set(g.astype(str)))
        pk = k.astype(str).value_counts(normalize=True).reindex(cats, fill_value=0)
        pg = g.astype(str).value_counts(normalize=True).reindex(cats, fill_value=0)
    else:
        # bin edges = the tenths of Kenya AND of Ghana, so a shift is visible even when most Kenyan values are equal
        # (95% of Kenyan loans are 7 days long, so Kenya's tenths alone would put 7 and 180 days in one bin)
        edges = np.unique(np.concatenate([np.nanquantile(k, np.linspace(0, 1, 11)), np.nanquantile(g, np.linspace(0, 1, 11))]))
        bins = lambda x: pd.Series(np.where(np.isnan(x), -1, np.searchsorted(edges[1:-1], x, side="right")))
        pk, pg = bins(k.to_numpy(float)).value_counts(normalize=True), bins(g.to_numpy(float)).value_counts(normalize=True)
        pk, pg = pk.reindex(pk.index.union(pg.index), fill_value=0), pg.reindex(pk.index.union(pg.index), fill_value=0)
    pk, pg = pk + eps, pg + eps
    return float(((pg - pk) * np.log(pg / pk)).sum())

rows = []
for c in FEATURES:
    cat = c in ["loan_type", "New_versus_Repeat"]
    r = {"clue": c, "PSI Kenya test": psi(tr[c], ke_test[c], cat), "PSI Ghana": psi(tr[c], gh[c], cat)}
    if c in AMOUNT_CLUES:
        r["PSI Ghana, in US$"] = psi(tr[c] / tr.usd_rate, gh[c] / gh.usd_rate)
    rows.append(r)
r = {"clue": "Total_Amount (Task 1 target)", "PSI Kenya test": psi(tr.Total_Amount, ke_test.Total_Amount),
     "PSI Ghana": psi(tr.Total_Amount, gh.Total_Amount), "PSI Ghana, in US$": psi(tr.Total_Amount / tr.usd_rate, gh.Total_Amount / gh.usd_rate)}
drift = pd.DataFrame(rows + [r]).set_index("clue")
drift["reading (Ghana)"] = pd.cut(drift[["PSI Ghana", "PSI Ghana, in US$"]].min(axis=1), [-1, .1, .25, np.inf], labels=["stable", "some shift", "big shift"])
drift.to_csv(TABLES / "10_psi.csv")
drift.round(3)

,PSI Kenya test,PSI Ghana,"PSI Ghana, in US$",reading (Ghana)
clue,,,,
loan_type,0.002,17.174,NaN,big shift
duration,0.000,0.642,NaN,big shift
New_versus_Repeat,0.000,0.008,NaN,stable
n_prev_loans,0.000,0.890,NaN,big shift
n_finished,0.001,7.181,NaN,big shift
n_defaults,0.000,0.004,NaN,stable
past_default_rate,0.000,8.887,NaN,big shift
no_known_history,0.000,8.890,NaN,big shift
days_since_last_loan,0.000,1.975,NaN,big shift


In [5]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.linspace(0, 12, 49)
for d, lab in [(tr, "Kenya (learned from)"), (gh, "Ghana")]:
    ax[0].hist(np.log(d.Total_Amount), bins=bins, density=True, alpha=.55, label=lab)
    ax[1].hist(np.log(d.Total_Amount / d.usd_rate), bins=np.linspace(-3, 10, 53), density=True, alpha=.55, label=lab)
ax[0].set(title="Loan amount, local currency", xlabel="log(amount): 7 ≈ 1,100, 9 ≈ 8,100", ylabel="share of loans")
ax[1].set(title="Loan amount, US dollars", xlabel="log(US$): 4 ≈ $55, 6 ≈ $400")
ax[0].legend(); plt.tight_layout(); fig.savefig(FIGURES / "10_amounts_kenya_vs_ghana.png", dpi=130)

![](../learnings/figures/10_amounts_kenya_vs_ghana.png)

## 2. Domain classifier: can the clues tell a Ghanaian applicant from a Kenyan one?

A **domain classifier** is a model trained to answer "is this loan from Ghana (1) or Kenya (0)?" from the clues alone. Its ROC-AUC says how different the two groups look:
- **0.5:** it can't tell them apart (the groups look alike);
- **1.0:** it can always tell (the groups don't overlap at all).

We train it on different sets of clues to see **which clues give Ghana away**. Tests keep each customer on one side (5 customer folds).

As a **control**, the same classifier tries to tell Kenya's labelled loans from Kenya's test loans. It should score close to 0.5.

In [6]:
REPAY = ["n_finished", "n_defaults", "past_default_rate", "no_known_history"]   # the outcome clues Ghana can't have
def with_usd(d):
    d = d.copy(); d[AMOUNT_CLUES] = d[AMOUNT_CLUES].div(d.usd_rate, axis=0); return d
SETS = {
    "all 13 clues (money in local currency)": (FEATURES, False),
    "all 13 clues (money in US$)": (FEATURES, True),
    "without loan_type": ([c for c in FEATURES if c != "loan_type"], True),
    "without outcome clues": ([c for c in FEATURES if c not in REPAY], True),
    "without loan_type, outcome clues": ([c for c in FEATURES if c not in REPAY + ["loan_type"]], True),
    "without loan_type, outcome clues, duration": ([c for c in FEATURES if c not in REPAY + ["loan_type", "duration"]], True),
    "loan_type only": (["loan_type"], True),
    "duration only": (["duration"], True),
    "outcome clues only": (REPAY, True),
    "money clues only (local currency)": (AMOUNT_CLUES, False),
    "money clues only (US$)": (AMOUNT_CLUES, True),
    "borrowing counts and speed only": (["n_prev_loans", "days_since_last_loan", "n_loans_last_7d", "n_loans_last_30d"], True),
}
def domain_oof(other, clues, usd, kind="logistic"):
    d = pd.concat([tr.assign(dom=0), other.assign(dom=1)], ignore_index=True)
    d = with_usd(d) if usd else d
    m = make_pipeline(preprocessor(clues), LogisticRegression(max_iter=3000, class_weight="balanced") if kind == "logistic"
                      else LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, random_state=0, is_unbalance=True))
    cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(d, d.dom, d.customer_id))
    return d, cross_val_predict(m, d[clues], d.dom, cv=cv, method="predict_proba")[:, 1]
rows, DOM = [], {}
for name, (clues, usd) in SETS.items():
    r = {"clues": name}
    for kind in ["logistic", "LightGBM"]:
        d, p = domain_oof(gh, clues, usd, kind); DOM[name, kind] = (d, p)
        r[f"AUC Ghana vs Kenya ({kind})"] = roc_auc_score(d.dom, p)
    d, p = domain_oof(ke_test, clues, usd, "LightGBM"); r["AUC control: Kenya test vs Kenya (LightGBM)"] = roc_auc_score(d.dom, p)
    rows.append(r)
dom = pd.DataFrame(rows).set_index("clues"); dom.to_csv(TABLES / "10_domain_classifier.csv")
dom.round(3)

,AUC Ghana vs Kenya (logistic),AUC Ghana vs Kenya (LightGBM),AUC control: Kenya test vs Kenya (LightGBM)
clues,,,
all 13 clues (money in local currency),1.000,1.000,0.502
all 13 clues (money in US$),1.000,1.000,0.503
without loan_type,0.992,0.994,0.503
without outcome clues,1.000,1.000,0.499
"without loan_type, outcome clues",0.945,0.973,0.502
"without loan_type, outcome clues, duration",0.939,0.948,0.500
loan_type only,1.000,1.000,0.498
duration only,0.543,0.586,0.500
outcome clues only,0.913,0.913,0.507


## 3. Importance weights: making Kenya look like Ghana

**The idea.** If some Kenyan loans look like Ghanaian ones, we can count those loans more when learning or testing. Then Kenya's data "looks like" Ghana. The weight of a Kenyan loan is

  weight = P(Ghana | clues) ÷ P(Kenya | clues) × (Kenyan loans ÷ Ghanaian loans),

taken from the domain classifier. A Kenyan loan that looks very Ghanaian gets a big weight; a typical Kenyan loan gets almost zero.

**The catch: the effective sample size (ESS).** ESS = (sum of weights)² ÷ (sum of squared weights). It says how many loans the weighted data is "really" worth. If a few loans carry all the weight, ESS collapses, and any weighted score rests on those few.

**Clipping** caps the biggest weights (a percentile cap, or a fixed number; the weights are then re-scaled to average 1, so the largest weight shown can end above the cap). That gives back some ESS, but makes Kenya look less like Ghana. We try several caps.

We use the **logistic** domain classifier on the clues Ghana can actually share with Kenya (no `loan_type`, no outcome clues, money in US$). With all 13 clues the groups barely overlap at all (section 2), so the weights there are meaningless.

In [7]:
def weights(name, kind="logistic", cap=None):
    d, p = DOM[name, kind]
    k = d.dom.to_numpy() == 0
    p = np.clip(p[k], 1e-6, 1 - 1e-6)
    w = p / (1 - p) * (k.sum() / (~k).sum())
    w = w / w.mean()                                   # average weight 1
    if cap is not None:
        w = np.minimum(w, np.quantile(w, cap) if cap < 1 else cap); w = w / w.mean()
    return pd.Series(w, index=tr.index)
def ess(w): return w.sum() ** 2 / (w ** 2).sum()
W_SET = "without loan_type, outcome clues"
CAPS = {"no cap": None, "cap at 99.9th percentile": .999, "cap at 99th percentile": .99, "cap at 50": 50, "cap at 10": 10, "cap at 5": 5}
rows = []
for set_name in [W_SET, "without loan_type, outcome clues, duration", "all 13 clues (money in US$)"]:
    for cap_name, cap in CAPS.items():
        w = weights(set_name, cap=cap)
        rows.append({"clues": set_name, "clipping": cap_name, "ESS (loans)": ess(w), "ESS % of Kenya": ess(w) / len(w),
                     "largest weight": w.max(), "weight held by top 1% of loans": np.sort(w)[::-1][: len(w) // 100].sum() / w.sum()})
wt = pd.DataFrame(rows); wt.to_csv(TABLES / "10_weights.csv", index=False)
d, p = DOM[W_SET, "logistic"]
print(f"Ghanaian loans the classifier is >99% sure are Ghanaian (no Kenyan lookalike): {(p[d.dom == 1] > .99).mean():.1%}")
wt.round(3)

Ghanaian loans the classifier is >99% sure are Ghanaian (no Kenyan lookalike): 11.8%


,clues,clipping,ESS (loans),ESS % of Kenya,largest weight,weight held by top 1% of loans
0,"without loan_type, outcome clues",no cap,18.777,0.000,8810.877,0.989
1,"without loan_type, outcome clues",cap at 99.9th percentile,250.118,0.004,429.479,0.951
2,"without loan_type, outcome clues",cap at 99th percentile,1933.766,0.028,52.758,0.527
3,"without loan_type, outcome clues",cap at 50,343.188,0.005,298.497,0.934
4,"without loan_type, outcome clues",cap at 10,708.321,0.010,136.012,0.849
5,"without loan_type, outcome clues",cap at 5,973.953,0.014,100.213,0.778
6,"without loan_type, outcome clues, duration",no cap,31.178,0.000,6517.508,0.971
7,"without loan_type, outcome clues, duration",cap at 99.9th percentile,314.791,0.005,368.401,0.906
8,"without loan_type, outcome clues, duration",cap at 99th percentile,2428.378,0.035,45.912,0.459
9,"without loan_type, outcome clues, duration",cap at 50,477.546,0.007,227.108,0.869


## 4. Task 1 on real Ghanaian amounts

This is the one place we have real Ghanaian answers. We learn the step 9 amount model on Kenya and predict every Ghanaian loan. We try:
- **currency:** feed cedis in as if they were shillings ("as is"), or convert everything to US dollars;
- **loan type:** keep `loan_type` (Ghana's Type_3 is then simply "unknown" to the model), or drop it;
- **weights:** learn with the importance weights at several caps, or without;
- **naive rules:** "guess Kenya's average", and "the same as your last loan" (for first loans: Ghana's own median, which we're actually not allowed to know, so this rule is a slight cheat).

Scored on the log scale, like step 9. **Bias** is the median of predicted ÷ real: 1.00 is unbiased, 0.5 means it predicts half.

In [8]:
from src.amounts import final_amount_model
def amount_scores(pred_local, d):
    e = np.log(d.Total_Amount.to_numpy()) - np.log(pred_local)
    return {"MAE log": np.abs(e).mean(), "within 2x": (np.abs(e) < np.log(2)).mean(), "bias (predicted / real)": float(np.median(np.exp(-e)))}
def amount_fit(clues, usd, w=None):
    from lightgbm import LGBMRegressor
    X = with_usd(tr) if usd else tr
    y = np.log(tr.Total_Amount / (tr.usd_rate if usd else 1))
    m = make_pipeline(preprocessor(clues), LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=0))
    m.fit(X[clues], y, **({} if w is None else {"lgbmregressor__sample_weight": w.to_numpy()}))
    G = with_usd(gh) if usd else gh
    return np.exp(m.predict(G[clues])) * (gh.usd_rate.to_numpy() if usd else 1)
no_type = [c for c in FEATURES if c != "loan_type"]
rows = [{"option": "guess Kenya's average (local currency)", **amount_scores(np.full(len(gh), np.exp(np.log(tr.Total_Amount).mean())), gh)},
        {"option": "guess Kenya's average (US$)", **amount_scores(np.full(len(gh), np.exp(np.log(tr.Total_Amount / tr.usd_rate).mean())) * gh.usd_rate.to_numpy(), gh)},
        {"option": "same as last loan (first loans: Ghana's median, a slight cheat)", **amount_scores(gh.last_amount.fillna(gh.Total_Amount.median()).to_numpy(), gh)}]
for label, clues, usd in [("model, as is (cedis fed in as shillings)", FEATURES, False), ("model, US$", FEATURES, True),
                          ("model, as is, no loan_type", no_type, False), ("model, US$, no loan_type", no_type, True)]:
    rows.append({"option": label, **amount_scores(amount_fit(clues, usd), gh)})
for cap_name in ["no cap", "cap at 50", "cap at 10"]:
    rows.append({"option": f"model, US$, no loan_type, weighted ({cap_name})", **amount_scores(amount_fit(no_type, True, weights(W_SET, cap=CAPS[cap_name])), gh)})
t1 = pd.DataFrame(rows).set_index("option"); t1.to_csv(TABLES / "10_amount_on_ghana.csv")
t1.round(3)

,MAE log,within 2x,bias (predicted / real)
option,,,
guess Kenya's average (local currency),1.087,0.391,1.760
guess Kenya's average (US$),2.381,0.011,0.127
"same as last loan (first loans: Ghana's median, a slight cheat)",0.522,0.836,0.992
"model, as is (cedis fed in as shillings)",0.453,0.804,1.234
"model, US$",0.509,0.764,0.820
"model, as is, no loan_type",0.507,0.793,0.987
"model, US$, no loan_type",0.626,0.709,0.633
"model, US$, no loan_type, weighted (no cap)",0.736,0.614,0.571
"model, US$, no loan_type, weighted (cap at 50)",0.724,0.618,0.567


**Kenya's own future-loan error, for comparison** (step 9, LightGBM): 0.58 on the log scale, about 1.8×. **By group in Ghana**, with the best option above. A group with a big bias would get limits that are systematically too high or too low.

In [9]:
best_opt = t1.drop(index=[i for i in t1.index if not i.startswith("model")])["MAE log"].idxmin()
print("best model option:", best_opt)
lab = best_opt
usd = "US$" in lab; clues = no_type if "no loan_type" in lab else FEATURES
w = weights(W_SET, cap=CAPS[lab.split("(")[1].rstrip(")")]) if "weighted" in lab else None
pred = amount_fit(clues, usd, w)
gh["_e"] = np.log(gh.Total_Amount) - np.log(pred)
groups = {"loan type": gh.loan_type, "loan length": pd.cut(gh.duration, [0, 7, 30, 90, 1000], labels=["7 days", "8-30", "31-90", "over 90"]),
          "first loan in the data": np.where(gh.n_prev_loans == 0, "first loan", "earlier loans")}
rows = []
for g, s in groups.items():
    for v, e in gh["_e"].groupby(s):
        if len(e) >= 30:
            rows.append({"group": g, "value": v, "loans": len(e), "MAE log": e.abs().mean(), "bias (predicted / real)": float(np.median(np.exp(-e)))})
bygroup = pd.DataFrame(rows); bygroup.to_csv(TABLES / "10_amount_on_ghana_by_group.csv", index=False)
gh.drop(columns="_e", inplace=True)
bygroup.round(3)

best model option: model, as is (cedis fed in as shillings)


,group,value,loans,MAE log,bias (predicted / real)
0,loan type,Type_2,449,1.041,2.819
1,loan type,Type_3,3039,0.330,1.169
2,loan type,Type_8,37,3.471,42.333
3,loan length,7 days,3041,0.331,1.169
4,loan length,over 90,479,1.229,3.125
5,first loan in the data,earlier loans,2863,0.260,1.149
6,first loan in the data,first loan,662,1.290,3.387


## 5. Task 2: how the Kenyan default model treats Ghana

There are no Ghanaian labels, so we can't measure accuracy here. What we *can* see is how the model scores Ghanaian applicants under each option. The **share declined** uses Ghana's cut-off (3.2%) on the calibrated probability (ADR 0007).

The options are the same paths as above, plus what to do with **unseen loan types** (Type_3, Type_8):
- **as is:** the model's one-hot step ignores an unknown type, so these loans get no loan-type push at all;
- **map to Type_1:** treat them as Kenya's most common type (also 7-day, small loans);
- **drop `loan_type`** from the model entirely.

For each option, the **Kenya future-loan lift** shows what it costs where we *can* measure. **Lift** = PR-AUC ÷ default rate, averaged over the 4 future-loan tests. The step 7 model gets about 8.

In [10]:
def calibrated_lr(train, clues, w=None):
    '''Step 7's final recipe (LR + Platt on out-of-fold scores from 5 customer folds), for any clues and optional weights.'''
    X, y = train[clues], train.target.astype(int)
    cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=17).split(X, y, train.customer_id))
    base = lambda: make_pipeline(preprocessor(clues), LogisticRegression(max_iter=2000))
    fp = {} if w is None else {"logisticregression__sample_weight": np.asarray(w)}
    oofp = np.zeros(len(X))
    for a, b in cv:
        oofp[b] = base().fit(X.iloc[a], y.iloc[a], **({} if w is None else {"logisticregression__sample_weight": np.asarray(w)[a]})).predict_proba(X.iloc[b])[:, 1]
    cal = LogisticRegression(max_iter=1000, random_state=17).fit(logit(oofp).reshape(-1, 1), y, **({} if w is None else {"sample_weight": np.asarray(w)}))
    m = base().fit(X, y, **fp)
    return lambda Z: cal.predict_proba(logit(m.predict_proba(Z[clues])[:, 1]).reshape(-1, 1))[:, 1]

# check: with all clues and no weights this is exactly step 7's final model
p_ours, p_step7 = calibrated_lr(tr, FEATURES)(gh), final_model(tr).predict_proba(gh[FEATURES])[:, 1]
assert np.allclose(p_ours, p_step7, atol=1e-6), "should match src/classification_experiments.final_model"

def future_lift(clues, X=None, w=None):
    X = tr if X is None else X; out = []
    for a, b in splits(tr, "time_block"):
        m = make_pipeline(preprocessor(clues), LogisticRegression(max_iter=2000))
        m.fit(X.loc[a, clues], tr.loc[a, "target"], **({} if w is None else {"logisticregression__sample_weight": w[a].to_numpy()}))
        y = tr.loc[b, "target"]; out.append(average_precision_score(y, m.predict_proba(X.loc[b, clues])[:, 1]) / y.mean())
    return np.mean(out)

gh_t1 = gh.assign(loan_type=gh.loan_type.where(gh.loan_type.isin(set(tr.loan_type)), "Type_1"))
first = gh.n_prev_loans.to_numpy() == 0
OPTS = [  # label, clues, money in US$, Ghana rows, weights
    ("as is (step 7 model)", FEATURES, False, gh, None),
    ("money in US$", FEATURES, True, gh, None),
    ("unseen types mapped to Type_1", FEATURES, False, gh_t1, None),
    ("unseen types mapped to Type_1, US$", FEATURES, True, gh_t1, None),
    ("drop loan_type", no_type, False, gh, None),
    ("drop loan_type, US$", no_type, True, gh, None),
    ("drop loan_type, US$, weighted (cap at 10)", no_type, True, gh, weights(W_SET, cap=10)),
    ("drop loan_type, US$, weighted (cap at 50)", no_type, True, gh, weights(W_SET, cap=50)),
]
rows, PD = [], {}
for label, clues, usd, G, w in OPTS:
    T, Gx = (with_usd(tr), with_usd(G)) if usd else (tr, G)
    p = calibrated_lr(T, clues, w)(Gx); PD[label] = p
    dec = p > CUTOFF["Ghana"]
    rows.append({"option": label, "Kenya future-loan lift": future_lift(clues, T, w),
                 "Ghana: average predicted default": p.mean(), "Ghana: share declined": dec.mean(),
                 "declined, first loans": dec[first].mean(), "declined, earlier loans": dec[~first].mean(),
                 **{f"declined, {t}": dec[(G.loan_type if 'mapped' not in label else gh.loan_type).to_numpy() == t].mean() for t in ["Type_3", "Type_2"]}})
t2 = pd.DataFrame(rows).set_index("option")
ke_p = final_model(tr).predict_proba(ke_test[FEATURES])[:, 1]
t2.loc["for comparison: Kenya test applicants, step 7 model, 2.1% cut-off", ["Ghana: average predicted default", "Ghana: share declined"]] = [ke_p.mean(), (ke_p > CUTOFF["Kenya"]).mean()]
t2.to_csv(TABLES / "10_default_model_on_ghana.csv")
t2.round(3)

,Kenya future-loan lift,Ghana: average predicted default,Ghana: share declined,"declined, first loans","declined, earlier loans","declined, Type_3","declined, Type_2"
option,,,,,,,
as is (step 7 model),7.775,0.125,0.474,1.000,0.353,0.390,1.0
money in US$,7.893,0.128,0.467,1.000,0.344,0.382,1.0
unseen types mapped to Type_1,7.775,0.108,0.154,0.601,0.050,0.018,1.0
"unseen types mapped to Type_1, US$",7.893,0.109,0.148,0.601,0.043,0.011,1.0
drop loan_type,6.885,0.087,0.283,1.000,0.117,0.168,1.0
"drop loan_type, US$",6.737,0.085,0.242,1.000,0.067,0.121,1.0
"drop loan_type, US$, weighted (cap at 10)",5.217,0.140,1.000,1.000,1.000,1.000,1.0
"drop loan_type, US$, weighted (cap at 50)",4.656,0.120,0.679,1.000,0.605,0.628,1.0
"for comparison: Kenya test applicants, step 7 model, 2.1% cut-off",NaN,0.018,0.192,NaN,NaN,NaN,NaN


## 6. The best Kenyan stand-ins for Ghana

Ghana's applicants are of two kinds: **first loans** (no earlier loans in the data, 19%) and **earlier loans with no known outcome** (81%). We find Kenyan loans of the same two kinds and score the step 7 model on them, on future loans. These are the closest measurable guesses of how well the model ranks Ghanaian risk, *if Ghanaians behaved like those Kenyans*.

The interval is a 95% customer bootstrap (1,000 re-draws).

In [11]:
o = []
for a, b in splits(tr, "time_block"):
    m = make_pipeline(preprocessor(FEATURES), LogisticRegression(max_iter=2000)).fit(tr.loc[a, FEATURES], tr.loc[a, "target"])
    o.append(pd.DataFrame({"fold": len(o), "p": m.predict_proba(tr.loc[b, FEATURES])[:, 1]}, index=b))
o = pd.concat(o).join(tr[["target", "customer_id", "n_prev_loans", "no_known_history"]])
def lift(g): return average_precision_score(g.target, g.p) / g.target.mean()
def boot_lift(g, n=1000, seed=0):
    rng, cs = np.random.default_rng(seed), g.customer_id.unique(); by = {c: i for c, i in g.groupby("customer_id").indices.items()}
    vals = []
    for _ in range(n):
        idx = np.concatenate([by[c] for c in rng.choice(cs, len(cs))]); s = g.iloc[idx]
        if s.target.sum() > 0: vals.append(lift(s))
    return np.percentile(vals, [2.5, 97.5])
groups = {"all future loans": o, "first loans (like Ghana's 19%)": o[o.n_prev_loans == 0],
          "earlier loans, no known outcome (like Ghana's 81%)": o[(o.n_prev_loans > 0) & (o.no_known_history == 1)],
          "earlier loans, known outcomes (no Ghanaian has these)": o[o.no_known_history == 0]}
rows = []
for name, g in groups.items():
    lo, hi = boot_lift(g)
    rows.append({"Kenyan loans": name, "loans": len(g), "defaults": int(g.target.sum()), "default rate": g.target.mean(),
                 "lift (pooled)": lift(g), "95% low": lo, "95% high": hi})
stand = pd.DataFrame(rows).set_index("Kenyan loans"); stand.to_csv(TABLES / "10_kenyan_standins.csv")
stand.round(3)

,loans,defaults,default rate,lift (pooled),95% low,95% high
Kenyan loans,,,,,,
all future loans,54463,988,0.018,7.064,6.097,8.427
first loans (like Ghana's 19%),2514,275,0.109,2.505,2.069,3.041
"earlier loans, no known outcome (like Ghana's 81%)",1436,69,0.048,6.449,4.451,10.382
"earlier loans, known outcomes (no Ghanaian has these)",50513,644,0.013,5.886,4.805,7.830


## 7. How many labelled Ghanaian loans would NkwaPay need?

Once Ghanaian loans start falling due, NkwaPay can measure the model there. But with only a few defaults that measure is very noisy. We imitate this with Kenyan future loans: draw a random sample of *n* loans many times, compute the lift each time, and see how wide the spread is.

In [12]:
rng, rows = np.random.default_rng(0), []
for n in [500, 1000, 2000, 5000, 10000, 20000]:
    vals, dfl = [], []
    for _ in range(300):
        s = o.iloc[rng.choice(len(o), n, replace=False)]
        if s.target.sum() >= 2: vals.append(lift(s)); dfl.append(s.target.sum())
    lo, hi = np.percentile(vals, [2.5, 97.5])
    rows.append({"labelled loans": n, "defaults (typical)": int(np.median(dfl)), "lift 95% low": lo, "lift 95% high": hi, "width": hi - lo})
need = pd.DataFrame(rows).set_index("labelled loans"); need.to_csv(TABLES / "10_sample_size.csv")
need.round(2)

,defaults (typical),lift 95% low,lift 95% high,width
labelled loans,,,,
500,9,3.01,32.81,29.80
1000,18,3.22,21.52,18.30
2000,36,4.46,15.19,10.73
5000,90,4.93,10.95,6.02
10000,182,5.71,9.15,3.43
20000,364,6.05,8.51,2.47


## 8. What this means

1. **Ghana is not "Kenya in another currency".** The domain classifier tells them apart almost perfectly (ROC-AUC 1.00 with all clues), while the Kenyan control scores 0.50. Most of the gap comes from the clues Ghana *can't* share:
   - `loan_type`: 87% of Ghanaian loans use a type never seen in Kenya;
   - the outcome clues: no Ghanaian has a known repayment.

   Even without those clues, the AUC is still 0.95–0.97.
2. **Importance weighting can't fix this.** The weighted Kenyan data is worth about 19 loans (ESS) without a cap, and at best about 2,800 loans (4%) with the strongest cap. Every weighted model was worse:
   - the Kenyan lift falls from 6.9 to 4.7–5.2;
   - the decline rate in Ghana jumps to 68–100%.

   **We don't use weights.**
3. **The default model, as it stands, would decline 47% of Ghanaian applicants**, against 19% in Kenya. That includes every first-time borrower and every Type_2 loan. The type effect is learned from 74 Kenyan Type_2 loans, 61% of which defaulted. Mapping unseen types to Type_1 brings the decline rate to 15%. Dropping `loan_type` gives 24–28%, and costs the Kenyan lift 7.8 → 6.9. None of these can be checked without Ghanaian outcomes.
4. **The amount model transfers surprisingly well** on real Ghanaian amounts, if cedis are fed in as they are: the log error is 0.45, better than Kenya's own 0.58. It mostly learns "about the size of your recent loans". Converting to dollars is *worse* (0.51), and weights are much worse (0.72–0.75). But it **over-predicts** for:
   - first loans: 3.4×;
   - loans over 90 days: 3.1×;
   - Type_2: 2.8×.
5. **The closest Kenyan stand-ins** suggest the model ranks Ghanaian-like borrowers less well:
   - first loans: lift 2.5 (95%: 2.1–3.0);
   - earlier loans with no known outcome: lift 6.5 (4.5–10.4, only 69 defaults);
   - all future loans, for comparison: 7.1.
6. **Evidence needed before trusting it in Ghana:** about **5,000 labelled Ghanaian loans** (~90 defaults) to know the lift within about ±3, and 10,000 (~180 defaults) for about ±1.7.

The full write-up is in [learnings/10](../learnings/10_transfer.md), and the decision in [ADR 0019](../docs/adr/0019-kenya-to-ghana-transfer.md).

# Part 2 · Fairness (step 11, Joseph, taking over from Liza)

**The question.** Who does the model treat worse, is that fair, and what should we change? (Brief Part D.) **Decision record:** [ADR 0020](../docs/adr/0020-fairness-mitigation.md). **Plain write-up:** [learnings/11](../learnings/11_fairness.md). **Audit documents:** [Model Card](../docs/model_card.md), [datasheet](../docs/datasheet.md), [compliance map](../docs/compliance.md).

**How we measure.** The main test is future loans: learn from older loans, score the next block, with step 7's final recipe. Only Kenya has outcomes, so Ghana gets approval rates only. "Positive" means flagged as risky, i.e. declined.

**Privacy:** every output is a count, a share or a score. Lenders are shown as A, B, C.

In [13]:
from sklearn.metrics import f1_score
from src.amounts import final_amount_model
from src.features import BORROW_OFTEN, DEFAULT_CLUES, GHANA_CLUES
from src.reasons import reasons, reference, pushes

FN, FP, FP_REPEAT = 4700, 100, 500  # KES per loan, ADR 0007; FP_REPEAT also counts lost repeat business

def recipe(train, clues=FEATURES, w=None, weighted_platt=True):
    """Step 7's final recipe for any clues or row weights. Returns (scorer, honest scores on the learning rows)."""
    X, y = train[clues], train.target.astype(int)
    w = None if w is None else np.asarray(w)
    base = lambda: make_pipeline(preprocessor(clues), LogisticRegression(max_iter=2000))
    oof = np.zeros(len(X))
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=17).split(X, y, train.customer_id):
        fit = {} if w is None else {"logisticregression__sample_weight": w[a]}
        oof[b] = base().fit(X.iloc[a], y.iloc[a], **fit).predict_proba(X.iloc[b])[:, 1]
    cal = LogisticRegression(max_iter=1000, random_state=17).fit(logit(oof).reshape(-1, 1), y, sample_weight=w if weighted_platt else None)
    m = base().fit(X, y, **({} if w is None else {"logisticregression__sample_weight": w}))
    honest = cal.predict_proba(logit(oof).reshape(-1, 1))[:, 1]
    return (lambda Z: cal.predict_proba(logit(m.predict_proba(Z[clues])[:, 1]).reshape(-1, 1))[:, 1]), honest

score_all, _ = recipe(tr)
assert np.allclose(score_all(gh), final_model(tr).predict_proba(gh[FEATURES])[:, 1], atol=1e-6), "should match step 7's final model"

def future(clues=FEATURES, weight=None, **kw):
    """Scores on future loans, block by block: learn from older loans, score the next block. Also returns each block's honest learning scores."""
    out, inner = [], []
    for k, (a, b) in enumerate(splits(tr, "time_block")):
        t = tr.loc[a]
        score, honest = recipe(t, clues, None if weight is None else weight(t), **kw)
        out.append(pd.DataFrame({"block": k, "p": score(tr.loc[b])}, index=b))
        inner.append(pd.DataFrame({"block": k, "p": honest}, index=a))
    return pd.concat(out), inner

def tenure(d):
    return pd.Series(np.select([d.n_prev_loans == 0, d.no_known_history == 1], ["first loan", "earlier loans, no known outcome"], "earlier loans, known outcomes"), index=d.index)

SIZE_EDGES = tr.Total_Amount.quantile([.25, .5, .75]).to_numpy()
SIZES = ["Q1 smallest", "Q2", "Q3", "Q4 largest"]
def size_band(d, edges=SIZE_EDGES):
    return pd.cut(d.Total_Amount, [-np.inf, *edges, np.inf], labels=SIZES).astype(str)

lenders = tr.lender_id.value_counts().index  # shown as A, B, C... by size: the IDs say nothing to a reader
LENDER = {l: f"Lender {'ABCDEFG'[i]}" for i, l in enumerate(lenders)}
common_types = tr.loan_type.value_counts().index[:8]

scores, _ = future()
fo = scores.join(tr[["target", "customer_id", "n_prev_loans", "no_known_history", "Total_Amount", "loan_type", "lender_id", "New_versus_Repeat", "disbursement_date"]])
fo["tenure"], fo["size"] = tenure(fo), size_band(fo)
fo["lender"] = fo.lender_id.map(LENDER)
fo["type"] = fo.loan_type.where(fo.loan_type.isin(common_types), "other types")
fo["first"] = fo.n_prev_loans == 0
fo["period"] = fo.disbursement_date.dt.year.astype(str)
print(len(fo), "future loans |", int(fo.target.sum()), "defaults |", fo.customer_id.nunique(), "customers")

54463 future loans | 988 defaults | 5718 customers


## 9. Audit the data first

Before any model: how many loans and defaults does each group have, and how often does it default? Real differences here limit what a fair model can do.

In [14]:
def audit(d):
    return pd.Series({"loans": len(d), "customers": d.customer_id.nunique(), "defaults": int(d.target.sum()), "default rate": d.target.mean()})
all_loans = tr.assign(tenure=tenure(tr), size=size_band(tr), lender=tr.lender_id.map(LENDER),
                      type=tr.loan_type.where(tr.loan_type.isin(common_types), "other types"))
counts = pd.concat({g: all_loans.groupby(g).apply(audit, include_groups=False) for g in ["tenure", "New_versus_Repeat", "size", "type", "lender"]})
counts.to_csv(TABLES / "11_group_counts.csv")
counts.round(3)

loans  customers  \
tenure            earlier loans, known outcomes    56534.0     4965.0   
                  earlier loans, no known outcome   5567.0     3073.0   
                  first loan                        6363.0     5663.0   
New_versus_Repeat New Loan                           566.0      566.0   
                  Repeat Loan                      67898.0     6325.0   
size              Q1 smallest                      17119.0     3640.0   
                  Q2                               17176.0     3486.0   
                  Q3                               17053.0     3161.0   
                  Q4 largest                       17116.0     2455.0   
type              Type_1                           61723.0     5142.0   
                  Type_10                            466.0       33.0   
                  Type_14                             99.0       83.0   
                  Type_4                            1235.0      318.0   
                  Type_5                            1521.0      216.0   
                  Type_6                             357.0      108.0   
                  Type_7                            2654.0      382.0   
                  Type_9                             190.0      153.0   
                  other types                        219.0      149.0   
lender            Lender A                         64568.0     5970.0   
                  Lender B                          3454.0      995.0   
                  Lender C                           254.0      205.0   
                  Lender D                           188.0      155.0   

                                                   defaults  default rate  
tenure            earlier loans, known outcomes       733.0         0.013  
                  earlier loans, no known outcome      96.0         0.017  
                  first loan                          429.0         0.067  
New_versus_Repeat New Loan                            119.0         0.210  
                  Repeat Loan                        1139.0         0.017  
size              Q1 smallest                         219.0         0.013  
                  Q2                                  253.0         0.015  
                  Q3                                  270.0         0.016  
                  Q4 largest                          516.0         0.030  
type              Type_1                              520.0         0.008  
                  Type_10                              12.0         0.026  
                  Type_14                              71.0         0.717  
                  Type_4                              152.0         0.123  
                  Type_5                              122.0         0.080  
                  Type_6                               38.0         0.106  
                  Type_7                              235.0         0.089  
                  Type_9                               36.0         0.189  
                  other types                          72.0         0.329  
lender            Lender A                            784.0         0.012  
                  Lender B                            316.0         0.091  
                  Lender C                             80.0         0.315  
                  Lender D                             78.0         0.415

## 10. How the model treats each group

For each group, at the 2.1% cut-off:
- **approval rate** (independence: does everyone get yes equally often?);
- **good payers wrongly declined** and **defaulters caught** (separation: do mistakes fall equally?);
- **predicted vs actual default**, PPV and NPV (sufficiency: does a score mean the same thing?);
- PR-AUC, ROC-AUC, lift and F1 (ranking within the group).

Ranges are customer bootstraps (200 re-draws). Groups under 30 defaults are marked: their numbers are mostly noise.

In [15]:
def measures(d):
    """Every measure for one group of future loans, at Kenya's 2.1% cut-off. 'Positive' = declined = flagged as risky."""
    y, p = d.target.to_numpy().astype(int), d.p.to_numpy()
    dec = p > CUTOFF["Kenya"]
    good, bad = y == 0, y == 1
    r = {"approval rate": 1 - dec.mean(), "wrongly declined (good payers)": dec[good].mean() if good.any() else np.nan,
         "defaulters caught": dec[bad].mean() if bad.any() else np.nan,
         "declined who would default (PPV)": y[dec].mean() if dec.any() else np.nan,
         "approved who repay (NPV)": 1 - y[~dec].mean() if (~dec).any() else np.nan,
         "F1": f1_score(y, dec) if bad.any() else np.nan,
         "predicted default": p.mean(), "actual default": y.mean(),
         "cost per loan (KES)": (FN * (bad & ~dec).sum() + FP * (good & dec).sum()) / len(y)}
    r["PR-AUC"] = average_precision_score(y, p) if bad.any() and good.any() else np.nan
    r["ROC-AUC"] = roc_auc_score(y, p) if bad.any() and good.any() else np.nan
    r["lift"] = r["PR-AUC"] / y.mean() if bad.any() else np.nan
    return r

CI_FOR = ["approval rate", "wrongly declined (good payers)", "defaulters caught", "F1", "PR-AUC"]
def by_group(d, by, n=200, min_defaults=30):
    """measures() per group, with a customer-bootstrap 95% range. Groups with few defaults are marked: their numbers are noise."""
    rows = []
    for name, g in d.groupby(by, observed=True):
        r = {"loans": len(g), "defaults": int(g.target.sum()), "enough defaults": g.target.sum() >= min_defaults,
             "score 25%": g.p.quantile(.25), "score median": g.p.median(), "score 75%": g.p.quantile(.75), **measures(g)}
        rng, cs, idx = np.random.default_rng(0), g.customer_id.unique(), g.groupby("customer_id").indices
        draws = []
        for _ in range(n):
            s = g.iloc[np.concatenate([idx[c] for c in rng.choice(cs, len(cs))])]
            draws.append([measures(s)[m] for m in CI_FOR])
        lo, hi = np.nanpercentile(np.array(draws, float), [2.5, 97.5], axis=0)
        r.update({f"{m} low": l for m, l in zip(CI_FOR, lo)} | {f"{m} high": h for m, h in zip(CI_FOR, hi)})
        rows.append({by if isinstance(by, str) else " x ".join(by): name if isinstance(by, str) else " x ".join(name), **r})
    return pd.DataFrame(rows).set_index(list(rows[0])[0])

SHOW = ["loans", "defaults", "approval rate", "wrongly declined (good payers)", "defaulters caught", "F1", "PR-AUC", "lift", "predicted default", "actual default"]
fair = {by: by_group(fo, by) for by in ["tenure", "New_versus_Repeat", "size", "type", "lender", "period"]}
fair["all"] = by_group(fo.assign(all="all future loans"), "all")
pd.concat(fair).to_csv(TABLES / "11_fairness_by_group.csv")
pd.concat(fair)[SHOW].round(3)

loans  defaults  \
tenure            earlier loans, known outcomes    50513       644   
                  earlier loans, no known outcome   1436        69   
                  first loan                        2514       275   
New_versus_Repeat New Loan                           387        48   
                  Repeat Loan                      54076       940   
size              Q1 smallest                      13866       190   
                  Q2                               14182       224   
                  Q3                               13374       182   
                  Q4 largest                       13041       392   
type              Type_1                           49000       452   
                  Type_10                            400        10   
                  Type_4                             989       112   
                  Type_5                             800        56   
                  Type_6                             326        27   
                  Type_7                            2654       235   
                  Type_9                             190        36   
                  other types                        104        60   
lender            Lender A                         51654       699   
                  Lender B                          2563       215   
                  Lender C                           246        74   
period            2022                             50440       587   
                  2023                              1239       143   
                  2024                              2784       258   
all               all future loans                 54463       988   

                                                   approval rate  \
tenure            earlier loans, known outcomes            0.917   
                  earlier loans, no known outcome          0.846   
                  first loan                               0.447   
New_versus_Repeat New Loan                                 0.003   
                  Repeat Loan                              0.899   
size              Q1 smallest                              0.928   
                  Q2                                       0.903   
                  Q3                                       0.901   
                  Q4 largest                               0.836   
type              Type_1                                   0.972   
                  Type_10                                  0.722   
                  Type_4                                   0.004   
                  Type_5                                   0.046   
                  Type_6                                   0.000   
                  Type_7                                   0.249   
                  Type_9                                   0.000   
                  other types                              0.019   
lender            Lender A                                 0.932   
                  Lender B                                 0.191   
                  Lender C                                 0.000   
period            2022                                     0.948   
                  2023                                     0.147   
                  2024                                     0.233   
all               all future loans                         0.893   

                                                   wrongly declined (good payers)  \
tenure            earlier loans, known outcomes                             0.079   
                  earlier loans, no known outcome                           0.131   
                  first loan                                                0.530   
New_versus_Repeat New Loan                                                  0.997   
                  Repeat Loan                                               0.093   
size              Q1 smallest                                               0.070   
    

**Where groups overlap** (Mitchell's intersectional results), and the size of the main gaps with 95% ranges.

In [16]:
inter = {"tenure x size": by_group(fo, ["tenure", "size"]), "tenure x lender": by_group(fo, ["tenure", "lender"]), "tenure x period": by_group(fo, ["tenure", "period"])}
pd.concat(inter).to_csv(TABLES / "11_fairness_intersections.csv")
pd.concat(inter)[SHOW].round(3)

loans  \
tenure x size   earlier loans, known outcomes x Q1 smallest    12490   
                earlier loans, known outcomes x Q2             12927   
                earlier loans, known outcomes x Q3             12769   
                earlier loans, known outcomes x Q4 largest     12327   
                earlier loans, no known outcome x Q1 smallest    504   
                earlier loans, no known outcome x Q2             407   
                earlier loans, no known outcome x Q3             267   
                earlier loans, no known outcome x Q4 largest     258   
                first loan x Q1 smallest                         872   
                first loan x Q2                                  848   
                first loan x Q3                                  338   
                first loan x Q4 largest                          456   
tenure x lender earlier loans, known outcomes x Lender A       48482   
                earlier loans, known outcomes x Lender B        1959   
                earlier loans, known outcomes x Lender C          72   
                earlier loans, no known outcome x Lender A      1289   
                earlier loans, no known outcome x Lender B       128   
                earlier loans, no known outcome x Lender C        19   
                first loan x Lender A                           1883   
                first loan x Lender B                            476   
                first loan x Lender C                            155   
tenure x period earlier loans, known outcomes x 2022           47615   
                earlier loans, known outcomes x 2023             853   
                earlier loans, known outcomes x 2024            2045   
                earlier loans, no known outcome x 2022          1230   
                earlier loans, no known outcome x 2023           100   
                earlier loans, no known outcome x 2024           106   
                first loan x 2022                               1595   
                first loan x 2023                                286   
                first loan x 2024                                633   

                                                               defaults  \
tenure x size   earlier loans, known outcomes x Q1 smallest         117   
                earlier loans, known outcomes x Q2                  121   
                earlier loans, known outcomes x Q3                  149   
                earlier loans, known outcomes x Q4 largest          257   
                earlier loans, no known outcome x Q1 smallest         6   
                earlier loans, no known outcome x Q2                 10   
                earlier loans, no known outcome x Q3                 10   
                earlier loans, no known outcome x Q4 largest         43   
                first loan x Q1 smallest                             67   
                first loan x Q2                                      93   
                first loan x Q3                                      23   
                first loan x Q4 largest                              92   
tenure x lender earlier loans, known outcomes x Lender A            495   
                earlier loans, known outcomes x Lender B            140   
                earlier loans, known outcomes x Lender C              9   
                earlier loans, no known outcome x Lender A           41   
                earlier loans, no known outcome x Lender B           18   
                earlier loans, no known outcome x Lender C           10   
                first loan x Lender A                               163   
                first loan x Lender B                                57   
                first loan x Lender C                                55   
tenure x period earlier loans, known outcomes x 2022                421   
                earlier loans, known outcomes x 2023                 59   
                earlier loans,

In [17]:
# A bootstrap of the gap itself: first loans minus loans with known outcomes.
def gap_ci(d, col, a, b, measure, n=1000):
    x, rng = [], np.random.default_rng(0)
    cs, idx = d.customer_id.unique(), d.groupby("customer_id").indices
    for _ in range(n):
        s = d.iloc[np.concatenate([idx[c] for c in rng.choice(cs, len(cs))])]
        A, B = s[s[col] == a], s[s[col] == b]
        x.append(measure(A) - measure(B))
    return np.mean(x), *np.percentile(x, [2.5, 97.5])
fpr = lambda g: (g.p[g.target == 0] > CUTOFF["Kenya"]).mean()
tpr = lambda g: (g.p[g.target == 1] > CUTOFF["Kenya"]).mean()
apr = lambda g: (g.p <= CUTOFF["Kenya"]).mean()
gaps = []
for label, col, a, b in [("first loan vs known outcomes", "tenure", "first loan", "earlier loans, known outcomes"),
                         ("no known outcome vs known outcomes", "tenure", "earlier loans, no known outcome", "earlier loans, known outcomes"),
                         ("largest vs smallest loans", "size", "Q4 largest", "Q1 smallest")]:
    for mname, f in [("approval rate", apr), ("wrongly declined", fpr), ("defaulters caught", tpr)]:
        gaps.append(dict(zip(["comparison", "measure", "gap", "95% low", "95% high"], (label, mname, *gap_ci(fo, col, a, b, f)))))
gaps = pd.DataFrame(gaps); gaps.to_csv(TABLES / "11_gaps.csv", index=False)
gaps.round(3)

,comparison,measure,gap,95% low,95% high
0,first loan vs known outcomes,approval rate,-0.469,-0.492,-0.448
1,first loan vs known outcomes,wrongly declined,0.451,0.427,0.475
2,first loan vs known outcomes,defaulters caught,0.272,0.198,0.345
3,no known outcome vs known outcomes,approval rate,-0.070,-0.101,-0.044
4,no known outcome vs known outcomes,wrongly declined,0.053,0.029,0.077
5,no known outcome vs known outcomes,defaulters caught,0.137,-0.030,0.291
6,largest vs smallest loans,approval rate,-0.092,-0.112,-0.073
7,largest vs smallest loans,wrongly declined,0.077,0.059,0.097
8,largest vs smallest loans,defaulters caught,0.476,0.396,0.555


## 11. Does a score mean the same thing for every group?

In [18]:
# Calibration by group: does 'predicted X%' mean X% for every group?
bins = [0, .005, .01, .021, .05, .1, 1]
cal = (fo.assign(score_band=pd.cut(fo.p, bins)).groupby(["tenure", "score_band"], observed=True)
       .agg(loans=("target", "size"), defaults=("target", "sum"), predicted=("p", "mean"), actual=("target", "mean")))
cal.to_csv(TABLES / "11_calibration_by_group.csv")
cal.round(3)

loans  defaults  predicted  \
tenure                          score_band                                  
earlier loans, known outcomes   (0.0, 0.005]   34648       193      0.002   
                                (0.005, 0.01]   7020        70      0.007   
                                (0.01, 0.021]   4628        81      0.014   
                                (0.021, 0.05]   2655       136      0.032   
                                (0.05, 0.1]      837        74      0.066   
                                (0.1, 1.0]       725        90      0.262   
earlier loans, no known outcome (0.0, 0.005]     639         8      0.002   
                                (0.005, 0.01]    390        10      0.008   
                                (0.01, 0.021]    186         9      0.017   
                                (0.021, 0.05]     63         3      0.026   
                                (0.05, 0.1]       56        12      0.078   
                                (0.1, 1.0]       102        27      0.207   
first loan                      (0.0, 0.005]       2         0      0.002   
                                (0.005, 0.01]    372        29      0.008   
                                (0.01, 0.021]    750        43      0.018   
                                (0.021, 0.05]    562        63      0.041   
                                (0.05, 0.1]      311        34      0.073   
                                (0.1, 1.0]       517       106      0.238   

                                               actual  
tenure                          score_band             
earlier loans, known outcomes   (0.0, 0.005]    0.006  
                                (0.005, 0.01]   0.010  
                                (0.01, 0.021]   0.018  
                                (0.021, 0.05]   0.051  
                                (0.05, 0.1]     0.088  
                                (0.1, 1.0]      0.124  
earlier loans, no known outcome (0.0, 0.005]    0.013  
                                (0.005, 0.01]   0.026  
                                (0.01, 0.021]   0.048  
                                (0.021, 0.05]   0.048  
                                (0.05, 0.1]     0.214  
                                (0.1, 1.0]      0.265  
first loan                      (0.0, 0.005]    0.000  
                                (0.005, 0.01]   0.078  
                                (0.01, 0.021]   0.057  
                                (0.021, 0.05]   0.112  
                                (0.05, 0.1]     0.109  
                                (0.1, 1.0]      0.205

## 12. Proxies: can the clues guess the group?

If they can, deleting a column changes little, because the model still sees the group through other clues. (`knowledge/fairness_reading.md` §4.) AUC: 0.5 = no idea, 1.0 = always right.

In [19]:
# Proxy check: can the clues guess the group? AUC from customer-grouped folds; 0.5 = no idea, 1.0 = always.
HISTORY_CLUES = [c for c in FEATURES if c not in ["loan_type", "duration", "New_versus_Repeat"]]
AMOUNT = ["mean_prev_amount", "last_amount"]
def proxy_auc(df, y, clues, model="logistic"):
    clf = LogisticRegression(max_iter=2000) if model == "logistic" else LGBMClassifier(n_estimators=100, learning_rate=.1, num_leaves=15, verbose=-1, random_state=0, n_jobs=4)
    cv = list(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(df[clues], y, df.customer_id))
    p = cross_val_predict(make_pipeline(preprocessor(clues), clf), df[clues], y, cv=cv, method="predict_proba")[:, 1]
    return roc_auc_score(y, p)

both = pd.concat([tr.assign(is_ghana=0), gh.assign(is_ghana=1)]).reset_index(drop=True)
TARGETS = {
    "Ghana vs Kenya": (both, both.is_ghana, {"all 13 clues": FEATURES, "without loan_type": [c for c in FEATURES if c != "loan_type"],
                                             "without loan_type and outcome clues": [c for c in FEATURES if c not in ["loan_type", "n_defaults", "past_default_rate", "no_known_history", "n_finished"]]}),
    "first loan": (tr, (tr.n_prev_loans == 0).astype(int), {"all 13 clues": FEATURES, "loan details only (no history)": ["loan_type", "duration", "New_versus_Repeat"],
                                                            "loan_type and duration only": ["loan_type", "duration"]}),
    "'New Loan' flag": (tr, (tr.New_versus_Repeat == "New Loan").astype(int), {"without the flag": [c for c in FEATURES if c != "New_versus_Repeat"],
                                                                               "history clues only": HISTORY_CLUES}),
    "Lender A vs others": (tr, (tr.lender_id == lenders[0]).astype(int), {"all 13 clues": FEATURES, "without loan_type": [c for c in FEATURES if c != "loan_type"], "loan_type only": ["loan_type"]}),
    "largest quarter of loans": (tr, (tr.Total_Amount > SIZE_EDGES[2]).astype(int), {"all 13 clues": FEATURES, "without the two amount clues": [c for c in FEATURES if c not in AMOUNT], "the two amount clues only": AMOUNT}),
}
prox = []
for name, (df, y, sets) in TARGETS.items():
    for label, clues in sets.items():
        prox.append({"group": name, "clues used": label, "logistic": proxy_auc(df, y, clues), "LightGBM": proxy_auc(df, y, clues, "lgbm")})
prox = pd.DataFrame(prox); prox.to_csv(TABLES / "11_proxies.csv", index=False)
prox.round(3)

,group,clues used,logistic,LightGBM
0,Ghana vs Kenya,all 13 clues,1.000,1.000
1,Ghana vs Kenya,without loan_type,0.991,0.994
2,Ghana vs Kenya,without loan_type and outcome clues,0.900,0.943
3,first loan,all 13 clues,1.000,1.000
4,first loan,loan details only (no history),0.614,0.616
5,first loan,loan_type and duration only,0.590,0.593
6,'New Loan' flag,without the flag,0.971,0.972
7,'New Loan' flag,history clues only,0.957,0.957
8,Lender A vs others,all 13 clues,0.984,0.984
9,Lender A vs others,without loan_type,0.913,0.950


In [20]:
# Single clues: which ones give each group away most? (AUC of that clue alone, logistic.)
single = pd.DataFrame({name: {c: proxy_auc(df, y, [c]) for c in FEATURES if c not in (["New_versus_Repeat"] if "New" in name else [])} for name, (df, y, _) in TARGETS.items()})
single.to_csv(TABLES / "11_proxies_single_clue.csv")
single.round(3)

,Ghana vs Kenya,first loan,'New Loan' flag,Lender A vs others,largest quarter of loans
loan_type,1.000,0.590,0.727,0.983,0.572
duration,0.543,0.553,0.607,0.767,0.570
New_versus_Repeat,0.504,0.542,NaN,0.521,0.503
n_prev_loans,0.639,1.000,0.957,0.763,0.510
n_finished,0.913,0.954,0.916,0.713,0.512
n_defaults,0.501,0.501,0.500,0.505,0.502
past_default_rate,0.913,0.954,0.916,0.589,0.504
no_known_history,0.913,0.954,0.916,0.583,0.502
days_since_last_loan,0.815,1.000,0.957,0.847,0.553
n_loans_last_7d,0.773,0.847,0.818,0.796,0.526


## 13. The fixes, compared

Every option on the same future loans, costed with our cost matrix, with settings chosen on older loans only. A **starter loan** gives a first-time applicant above the cut-off a small loan instead of a no. It shrinks the money at risk, but not the 200 KES of chasing.

In [21]:
# The mitigations, on the future-loan test. Every setting is chosen on older loans only.
def balanced_weights(t):
    """Reweighing: tenure (first loan or not) and outcome count as if they were unrelated."""
    g, y = (t.n_prev_loans == 0).astype(int), t.target.astype(int)
    cell = g * 2 + y
    return (g.map(g.value_counts(normalize=True)) * y.map(y.value_counts(normalize=True)) / cell.map(cell.value_counts(normalize=True))).to_numpy()

def group_cutoff(inner, rule):
    """For each block, the cut-off for first loans that meets the rule on that block's older (learning) loans."""
    cuts = {}
    for k, h in enumerate(inner):
        h = h.join(tr[["target", "n_prev_loans"]]); first = h.n_prev_loans == 0
        ts, rep = np.quantile(h.p[first], np.linspace(0, 1, 401)), h[~first]
        if rule == "equal wrongly declined":
            goal, have = (rep.p[rep.target == 0] > CUTOFF["Kenya"]).mean(), np.array([(h.p[first & (h.target == 0)] > t).mean() for t in ts])
        else:  # equal approval
            goal, have = (rep.p > CUTOFF["Kenya"]).mean(), np.array([(h.p[first] > t).mean() for t in ts])
        cuts[k] = float(ts[np.argmin(np.abs(have - goal))])
    return cuts

CHASE = 200  # KES per default, from step 4: the part of FN that doesn't shrink with the loan
def decide(s, cut=None, starter=None):
    """Who is refused outright and who gets a starter loan. cut: per-block cut-off for first loans. starter: its size, as a share of a typical loan."""
    d = s.join(fo[["target", "first"]])
    d["y"] = d.target.astype(bool)
    t = d.block.map(cut).where(d["first"], CUTOFF["Kenya"]).astype(float) if cut else CUTOFF["Kenya"]
    d["over"] = d.p > t
    d["small"] = d.over & d["first"] if starter else False
    d["refused"] = d.over & ~d.small
    d["st"] = starter or 0
    return d

def loan_costs(d, fp=FP):
    """KES lost on each loan. A starter default loses its smaller amount plus the same chasing; a starter repaid earns part of the fee."""
    return (FN * (d.y & ~d.over) + fp * (~d.y & d.refused) + (d.st * (FN - CHASE) + CHASE) * (d.y & d.small) + (1 - d.st) * FP * (~d.y & d.small)).astype(float)

def judge(name, s, **kw):
    d = decide(s, **kw); y, f = d.y, d["first"]
    return {"option": name, "Kenya future-loan lift": np.mean([average_precision_score(g.target, g.p) / g.target.mean() for _, g in d.groupby("block")]),
            "cost per loan (KES)": loan_costs(d).mean(), "cost with repeat business (KES)": loan_costs(d, FP_REPEAT).mean(),
            "full approval": (~d.over).mean(), "any loan": (~d.refused).mean(),
            "first loans: any loan": (~d.refused)[f].mean(), "other loans: any loan": (~d.refused)[~f].mean(),
            "first loans: good payers refused": d.refused[f & ~y].mean(), "other loans: good payers refused": d.refused[~f & ~y].mean(),
            "first loans: defaulters stopped": d.refused[f & y].mean(), "other loans: defaulters stopped": d.refused[~f & y].mean(),
            "first loans: predicted / actual": d.p[f].mean() / y[f].mean(), "other loans: predicted / actual": d.p[~f].mean() / y[~f].mean()}

base, fut_inner = scores, future()[1]
no_often = future(DEFAULT_CLUES)[0]
HIDE_TENURE = [c for c in FEATURES if c not in ["New_versus_Repeat", "n_prev_loans", "no_known_history"]]
cut_fpr, cut_apr = group_cutoff(fut_inner, "equal wrongly declined"), group_cutoff(fut_inner, "equal approval")
OPTS2 = [("no fix", base, {}),
         ("first loans: starter loan, 10% of a typical loan", base, {"starter": .10}),
         ("first loans: starter loan, 25% of a typical loan", base, {"starter": .25}),
         ("first loans: starter loan, 50% of a typical loan", base, {"starter": .50}),
         ("first loans: own cut-off, equal wrongly declined", base, {"cut": cut_fpr}),
         ("first loans: own cut-off, equal approval", base, {"cut": cut_apr}),
         ("drop the 'borrow often' clues", no_often, {}),
         ("hide tenure (no New_versus_Repeat, n_prev_loans, no_known_history)", future(HIDE_TENURE)[0], {}),
         ("reweigh tenure and outcome, honest Platt", future(weight=balanced_weights, weighted_platt=False)[0], {}),
         ("reweigh tenure and outcome, weighted Platt", future(weight=balanced_weights)[0], {}),
         ("drop loan_type (ADR 0019's Ghana model)", future([c for c in FEATURES if c != "loan_type"])[0], {}),
         ("CHOSEN: drop 'borrow often' + starter loan at 10%", no_often, {"starter": .10})]
mit = pd.DataFrame([judge(n, s, **k) for n, s, k in OPTS2]).set_index("option")
mit.to_csv(TABLES / "11_mitigations.csv")
print("first-loan cut-offs, equal wrongly declined:", {k: round(v, 3) for k, v in cut_fpr.items()}, "| equal approval:", {k: round(v, 3) for k, v in cut_apr.items()})
mit.round(3).T

first-loan cut-offs, equal wrongly declined: {0: 0.016, 1: 0.017, 2: 0.055, 3: 0.065} | equal approval: {0: 0.062, 1: 0.043, 2: 0.06, 3: 0.088}


option,no fix,"first loans: starter loan, 10% of a typical loan","first loans: starter loan, 25% of a typical loan","first loans: starter loan, 50% of a typical loan","first loans: own cut-off, equal wrongly declined","first loans: own cut-off, equal approval",drop the 'borrow often' clues,"hide tenure (no New_versus_Repeat, n_prev_loans, no_known_history)","reweigh tenure and outcome, honest Platt","reweigh tenure and outcome, weighted Platt",drop loan_type (ADR 0019's Ghana model),CHOSEN: drop 'borrow often' + starter loan at 10%
Kenya future-loan lift,7.779,7.779,7.779,7.779,7.779,7.779,7.715,8.174,7.137,7.137,6.893,7.715
cost per loan (KES),47.930,50.135,52.324,55.972,52.265,52.674,47.342,47.748,51.955,52.303,51.789,49.509
cost with repeat business (KES),86.730,80.217,82.406,86.055,87.694,87.325,85.173,88.583,100.296,102.385,106.373,79.012
full approval,0.893,0.893,0.893,0.893,0.903,0.905,0.895,0.888,0.869,0.865,0.854,0.895
any loan,0.893,0.919,0.919,0.919,0.903,0.905,0.895,0.888,0.869,0.865,0.854,0.920
first loans: any loan,0.447,1.000,1.000,1.000,0.654,0.698,0.470,0.405,0.677,0.677,0.351,1.000
other loans: any loan,0.915,0.915,0.915,0.915,0.915,0.915,0.916,0.911,0.879,0.874,0.878,0.916
first loans: good payers refused,0.530,0.000,0.000,0.000,0.325,0.278,0.506,0.573,0.300,0.300,0.631,0.000
other loans: good payers refused,0.080,0.080,0.080,0.080,0.080,0.080,0.078,0.083,0.115,0.120,0.117,0.078
first loans: defaulters stopped,0.738,0.000,0.000,0.000,0.520,0.495,0.724,0.771,0.505,0.505,0.796,0.000


In [22]:
# Is a difference in cost real? Customer bootstrap of each option's cost per loan minus no fix.
cs, inv = np.unique(fo.customer_id.to_numpy(), return_inverse=True)
def boot_diff(a, b, n=1000):
    rng = np.random.default_rng(0); diff, cnt = np.bincount(inv, (a - b).reindex(fo.index).to_numpy(), len(cs)), np.bincount(inv, minlength=len(cs))
    x = [(w * diff).sum() / (w * cnt).sum() for w in rng.multinomial(len(cs), np.full(len(cs), 1 / len(cs)), n)]
    return (a.mean() - b.mean(), *np.percentile(x, [2.5, 97.5]))
rows = []
for n, s, k in OPTS2[1:]:
    for label, fp in [("main costs", FP), ("with repeat business", FP_REPEAT)]:
        rows.append(dict(zip(["option", "costs", "cost change (KES per loan)", "95% low", "95% high"], (n, label, *boot_diff(loan_costs(decide(s, **k), fp), loan_costs(decide(base), fp))))))
mit_ci = pd.DataFrame(rows); mit_ci.to_csv(TABLES / "11_mitigation_cost_change.csv", index=False)
mit_ci.round(1)

,option,costs,cost change (KES per loan),95% low,95% high
0,"first loans: starter loan, 10% of a typical loan",main costs,2.2,1.8,2.6
1,"first loans: starter loan, 10% of a typical loan",with repeat business,-6.5,-7.3,-5.8
2,"first loans: starter loan, 25% of a typical loan",main costs,4.4,3.6,5.3
3,"first loans: starter loan, 25% of a typical loan",with repeat business,-4.3,-5.3,-3.3
4,"first loans: starter loan, 50% of a typical loan",main costs,8.0,6.6,9.6
5,"first loans: starter loan, 50% of a typical loan",with repeat business,-0.7,-2.3,1.0
6,"first loans: own cut-off, equal wrongly declined",main costs,4.3,2.7,6.0
7,"first loans: own cut-off, equal wrongly declined",with repeat business,1.0,-0.7,2.7
8,"first loans: own cut-off, equal approval",main costs,4.7,3.1,6.6
9,"first loans: own cut-off, equal approval",with repeat business,0.6,-1.1,2.5


**What people get told.** Does dropping the clues remove the 'borrow more often' message? Are the reasons still stable?

In [23]:
# What do borrowers get told under each model? Top reason, in words, among those declined; numbers replaced by N.
import re
def told(clues, d, cut, label):
    m = final_model(tr, clues); p = m.predict_proba(d[FEATURES])[:, 1]; D = d[p > cut]
    r = reasons(m, D[FEATURES], reference(m, tr[FEATURES]), known_types=set(tr.loan_type))
    top = pd.Series([re.sub(r"\d[\d,]*", "N", x[0]) for x in r]).value_counts(normalize=True)
    often = np.mean([any("regular borrowers" in s or "a long gap" in s for s in x) for x in r])
    return top.rename(label), {"model": label, "declined": (p > cut).mean(), "reasons per person": np.mean([len(x) for x in r]), "told to borrow more often (any reason)": often}
pairs = [told(FEATURES, ke_test, CUTOFF["Kenya"], "Kenya, 13 clues"), told(DEFAULT_CLUES, ke_test, CUTOFF["Kenya"], "Kenya, no 'borrow often'"),
         told([c for c in FEATURES if c != "loan_type"], gh, CUTOFF["Ghana"], "Ghana, no loan_type"), told(GHANA_CLUES, gh, CUTOFF["Ghana"], "Ghana, no loan_type, no 'borrow often'")]
told_top = pd.concat([a for a, _ in pairs], axis=1).fillna(0); told_top.to_csv(TABLES / "11_reasons_top.csv")

def stability(clues):
    """Same top-3 themes when the model is refitted on each block's older loans? Declined Kenyan test applicants, the same people each time."""
    m = final_model(tr, clues); D = ke_test[m.predict_proba(ke_test[FEATURES])[:, 1] > CUTOFF["Kenya"]]
    def tops(mm):
        t = pushes(mm, D[FEATURES], reference(mm, tr[FEATURES]), "theme")
        return [frozenset(s[s >= .2].sort_values(ascending=False).index[:3]) for _, s in t.iterrows()]
    first = tops(m)
    return np.mean([np.mean([x == y for x, y in zip(first, tops(final_model(tr.loc[a], clues)))]) for a, _ in splits(tr, "time_block")])
told_sum = pd.DataFrame([b for _, b in pairs]).set_index("model")
told_sum.loc["Kenya, 13 clues", "same top 3 after refitting"] = stability(FEATURES)
told_sum.loc["Kenya, no 'borrow often'", "same top 3 after refitting"] = stability(DEFAULT_CLUES)
told_sum.to_csv(TABLES / "11_reasons_summary.csv")
display(told_sum.round(3))
told_top.round(3)

,declined,reasons per person,told to borrow more often (any reason),same top 3 after refitting
model,,,,
"Kenya, 13 clues",0.192,1.616,0.310,0.905
"Kenya, no 'borrow often'",0.179,1.617,0.000,0.854
"Ghana, no loan_type",0.283,1.555,0.336,NaN
"Ghana, no loan_type, no 'borrow often'",0.222,1.695,0.000,NaN


,"Kenya, 13 clues","Kenya, no 'borrow often'","Ghana, no loan_type","Ghana, no loan_type, no 'borrow often'"
"This is your first loan with us, so we have no borrowing history to go on.",0.443,0.474,0.302,0.38
Loans of this kind (Type_N) have been repaid less often than others.,0.223,0.308,0.000,0.00
You took no loans in the last N days; regular borrowers have repaid more reliably.,0.126,0.000,0.216,0.00
"You have only N earlier loan with us, so we know little about how you repay.",0.122,0.095,0.000,0.00
The loan is for N days; longer loans are repaid less often.,0.084,0.099,0.482,0.62
Your last loan was N days ago; a long gap gives us less recent information.,0.002,0.000,0.000,0.00
You did not repay N earlier loan.,0.000,0.000,0.000,0.00
"You have only N earlier loans with us, so we know little about how you repay.",0.000,0.024,0.000,0.00


In [24]:
# Before and after, group by group, with the model we choose (no 'borrow often' clues; the starter loan doesn't change scores).
fo_after = fo.assign(p=no_often.p)
after = {by: by_group(fo_after, by) for by in ["tenure", "size", "lender"]}
pd.concat(after).to_csv(TABLES / "11_fairness_after.csv")
pd.concat({"before": pd.concat({k: fair[k] for k in after}), "after": pd.concat(after)})[["approval rate", "wrongly declined (good payers)", "defaulters caught", "predicted default", "actual default"]].round(3)

approval rate  \
before tenure earlier loans, known outcomes            0.917   
              earlier loans, no known outcome          0.846   
              first loan                               0.447   
       size   Q1 smallest                              0.928   
              Q2                                       0.903   
              Q3                                       0.901   
              Q4 largest                               0.836   
       lender Lender A                                 0.932   
              Lender B                                 0.191   
              Lender C                                 0.000   
after  tenure earlier loans, known outcomes            0.919   
              earlier loans, no known outcome          0.818   
              first loan                               0.470   
       size   Q1 smallest                              0.948   
              Q2                                       0.905   
              Q3                                       0.896   
              Q4 largest                               0.829   
       lender Lender A                                 0.938   
              Lender B                                 0.122   
              Lender C                                 0.000   

                                               wrongly declined (good payers)  \
before tenure earlier loans, known outcomes                             0.079   
              earlier loans, no known outcome                           0.131   
              first loan                                                0.530   
       size   Q1 smallest                                               0.070   
              Q2                                                        0.088   
              Q3                                                        0.093   
              Q4 largest                                                0.148   
       lender Lender A                                                  0.063   
              Lender B                                                  0.800   
              Lender C                                                  1.000   
after  tenure earlier loans, known outcomes                             0.076   
              earlier loans, no known outcome                           0.157   
              first loan                                                0.506   
       size   Q1 smallest                                               0.051   
              Q2                                                        0.086   
              Q3                                                        0.098   
              Q4 largest                                                0.155   
       lender Lender A                                                  0.058   
              Lender B                                                  0.872   
              Lender C                                                  1.000   

                                               defaulters caught  \
before tenure earlier loans, known outcomes                0.466   
              earlier loans, no known outcome              0.609   
              first loan                                   0.738   
       size   Q1 smallest                                  0.211   
              Q2                                           0.629   
              Q3                                           0.516   
              Q4 largest                                   0.689   
       lender Lender A                                     0.395   
              Lender B                                     0.907   
              Lender C                                     1.000   
after  tenure earlier loans, known outcomes                0.470   
              earlier loans, no known outcome              0.681   
              first loan                                   0.724   
       size   Q1 smallest                       

## 14. Applicants the model has never seen

The model fitted on all labelled loans, on Zindi's test applicants. This is what would really happen on launch.

In [25]:
# The same choices on applicants the models never saw. No Ghanaian outcomes, so this shows only who gets what.
gh_first = gh.n_prev_loans.to_numpy() == 0
routed = gh_first | gh.loan_type.isin(["Type_2", "Type_8"]).to_numpy() | (gh.duration > 90).to_numpy()
gh_size = size_band(gh, gh.Total_Amount.quantile([.25, .5, .75]).to_numpy()).to_numpy()
ke_first, ke_size = ke_test.n_prev_loans.to_numpy() == 0, size_band(ke_test).to_numpy()
rows = []
def policy(name, p, cut, first, size, route=None):
    dec, rt = p > cut, np.zeros(len(p), bool) if route is None else route
    rows.append({"policy": name, "full loan": (~dec & ~rt).mean(), "starter or review": rt.mean(), "declined": (dec & ~rt).mean(),
                 "first loans: declined": (dec & ~rt)[first].mean(), "earlier loans: declined": (dec & ~rt)[~first].mean(),
                 **{f"{b}: declined": (dec & ~rt)[size == b].mean() for b in SIZES}})
for clues, name in [(FEATURES, "13 clues"), (DEFAULT_CLUES, "no 'borrow often'")]:
    p = recipe(tr, clues)[0](ke_test)
    policy(f"Kenya, {name}", p, CUTOFF["Kenya"], ke_first, ke_size)
    policy(f"Kenya, {name} + starter for first loans", p, CUTOFF["Kenya"], ke_first, ke_size, ke_first & (p > CUTOFF["Kenya"]))
for clues, name in [(FEATURES, "13 clues"), ([c for c in FEATURES if c != "loan_type"], "no loan_type"), (GHANA_CLUES, "no loan_type, no 'borrow often'")]:
    p = recipe(tr, clues)[0](gh)
    policy(f"Ghana, {name}", p, CUTOFF["Ghana"], gh_first, gh_size)
    policy(f"Ghana, {name} + routing (ADR 0019)", p, CUTOFF["Ghana"], gh_first, gh_size, routed)
gh_pol = pd.DataFrame(rows).set_index("policy"); gh_pol.to_csv(TABLES / "11_country_policies.csv")
gh_pol.round(3).T

policy,"Kenya, 13 clues","Kenya, 13 clues + starter for first loans","Kenya, no 'borrow often'","Kenya, no 'borrow often' + starter for first loans","Ghana, 13 clues","Ghana, 13 clues + routing (ADR 0019)","Ghana, no loan_type","Ghana, no loan_type + routing (ADR 0019)","Ghana, no loan_type, no 'borrow often'","Ghana, no loan_type, no 'borrow often' + routing (ADR 0019)"
full loan,0.808,0.808,0.821,0.821,0.526,0.526,0.717,0.717,0.778,0.778
starter or review,0.000,0.091,0.000,0.091,0.000,0.222,0.000,0.222,0.000,0.222
declined,0.192,0.101,0.179,0.088,0.474,0.252,0.283,0.061,0.222,0.000
first loans: declined,1.000,0.000,1.000,0.000,1.000,0.000,1.000,0.000,0.998,0.000
earlier loans: declined,0.111,0.111,0.097,0.097,0.353,0.311,0.117,0.075,0.042,0.000
Q1 smallest: declined,0.175,0.067,0.147,0.039,0.609,0.425,0.318,0.134,0.185,0.000
Q2: declined,0.174,0.075,0.162,0.063,0.406,0.319,0.167,0.079,0.087,0.000
Q3: declined,0.178,0.104,0.171,0.097,0.235,0.193,0.066,0.024,0.042,0.000
Q4 largest: declined,0.243,0.161,0.239,0.157,0.647,0.073,0.581,0.007,0.573,0.000


## 15. Is the loan limit regressive?

The limit (ADR 0018) against the real amount, by loan size and tenure, on future loans.

In [26]:
# Is the loan limit regressive? Limit vs the real amount, on future loans, for every block's model.
lim = []
for k, (a, b) in enumerate(splits(tr, "time_block")):
    m = final_amount_model(tr.loc[a]); part = tr.loc[b]
    lim.append(pd.DataFrame({"limit": m.limit(part)}, index=b))
lim = pd.concat(lim).join(fo[["Total_Amount", "tenure", "size", "p"]])
lim["fits"], lim["ratio"] = lim.Total_Amount <= lim.limit, lim.limit / lim.Total_Amount
lim_t = pd.concat({g: lim.groupby(g).agg(loans=("fits", "size"), fits=("fits", "mean"), limit_vs_real=("ratio", "median"), median_limit=("limit", "median"), median_real=("Total_Amount", "median")) for g in ["size", "tenure"]})
lim_t.to_csv(TABLES / "11_limit_by_group.csv")
lim_t.round(2)

loans  fits  limit_vs_real  \
size   Q1 smallest                      13866  0.96           2.78   
       Q2                               14182  0.86           1.59   
       Q3                               13374  0.65           1.15   
       Q4 largest                       13041  0.48           0.97   
tenure earlier loans, known outcomes    50513  0.74           1.39   
       earlier loans, no known outcome   1436  0.80           1.58   
       first loan                        2514  0.82           2.16   

                                        median_limit  median_real  
size   Q1 smallest                           3442.51       1299.0  
       Q2                                    6009.51       3845.0  
       Q3                                    8700.21       7488.0  
       Q4 largest                           22463.61      21273.0  
tenure earlier loans, known outcomes         7600.57       5198.0  
       earlier loans, no known outcome       5735.28       3990.0  
       first loan                            8284.16       4479.5

## 16. Who can't get credit in Ghana? (Findex 2021)

Survey-weighted shares with 95% ranges; aggregates only. Checked against published figures: all adults 68% any account and 60% mobile money, men 74.2% any account. The file has no region column, so region can't be reported.

In [27]:
fx = pd.read_csv(ROOT / "data" / "worldbank" / "micro_gha.csv", low_memory=False)
yes = lambda c: (fx[c] == 1).astype(float)
fx["formal or mobile loan"], fx["family or friends loan"] = yes("fin22a"), yes("fin22b")
fx["own a mobile phone"] = yes("mobileowner")
FX = {"any account": "account", "mobile money account": "account_mob", "own a mobile phone": "own a mobile phone", "borrowed at all": "borrowed", "formal or mobile loan": "formal or mobile loan", "family or friends loan": "family or friends loan", "saved": "saved"}
fx_groups = {"all adults": fx.index == fx.index, "women": fx.female == 1, "men": fx.female == 2, "poorest 40%": fx.inc_q <= 2, "richest 60%": fx.inc_q >= 3,
             "age 15-24": fx.age < 25, "age 25-44": fx.age.between(25, 44), "age 45+": fx.age >= 45, "rural": fx.urbanicity_f2f == 1, "urban": fx.urbanicity_f2f == 2,
             "primary or less": fx.educ == 1, "secondary or more": fx.educ.isin([2, 3])}
def wmean(x, w): return np.average(x, weights=w)
out, rng = [], np.random.default_rng(0)
for g, mask in fx_groups.items():
    sub = fx[mask]; r = {"group": g, "people asked": len(sub), "share of adults": sub.wgt.sum() / fx.wgt.sum()}
    for label, c in FX.items():
        v, w = sub[c].to_numpy(float), sub.wgt.to_numpy()
        ok = ~np.isnan(v); draws = [wmean(v[ok][i], w[ok][i]) for i in (rng.integers(0, ok.sum(), ok.sum()) for _ in range(500))]
        r[label] = wmean(v[ok], w[ok]); r[label + " low"], r[label + " high"] = np.percentile(draws, [2.5, 97.5])
    out.append(r)
findex = pd.DataFrame(out).set_index("group"); findex.to_csv(TABLES / "11_findex.csv")
findex[["people asked", "share of adults", *FX]].round(3)

,people asked,share of adults,any account,mobile money account,own a mobile phone,borrowed at all,formal or mobile loan,family or friends loan,saved
group,,,,,,,,,
all adults,1000,1.000,0.682,0.597,0.853,0.522,0.072,0.386,0.686
women,465,0.513,0.626,0.550,0.812,0.479,0.056,0.365,0.632
men,535,0.487,0.742,0.646,0.897,0.567,0.090,0.407,0.742
poorest 40%,309,0.399,0.547,0.464,0.794,0.521,0.049,0.405,0.579
richest 60%,691,0.601,0.772,0.685,0.892,0.523,0.088,0.373,0.757
age 15-24,291,0.300,0.652,0.616,0.838,0.495,0.045,0.380,0.709
age 25-44,549,0.419,0.727,0.666,0.892,0.546,0.075,0.413,0.743
age 45+,156,0.277,0.650,0.474,0.820,0.507,0.100,0.341,0.567
rural,360,0.401,0.565,0.479,0.781,0.520,0.066,0.360,0.630


In [28]:
# Two figures for the report.
show = pd.concat([fair["tenure"], fair["size"], fair["lender"]])
order = ["earlier loans, known outcomes", "earlier loans, no known outcome", "first loan", *SIZES, "Lender A", "Lender B", "Lender C"]
show = show.loc[[o for o in order if o in show.index]].iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.4)); yy = np.arange(len(show)); m = "wrongly declined (good payers)"
ax.hlines(yy, show[m + " low"], show[m + " high"], color="#0072B2", lw=2)
ax.plot(show[m], yy, "o", color="#0072B2", ms=8)
for i, v in enumerate(show[m]): ax.annotate(f"{v:.0%}", (show[m + " high"].iloc[i], i), xytext=(6, -3), textcoords="offset points", fontsize=9)
ax.axvline(fair["all"][m].iloc[0], color="#777777", lw=1, ls="--"); ax.annotate("all loans", (fair["all"][m].iloc[0], len(show) - .4), xytext=(4, 0), textcoords="offset points", fontsize=9, color="#555555")
ax.set_yticks(yy, show.index); ax.set_xlim(0, 1.1); ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))
ax.set_title("Good payers wrongly declined, future Kenyan loans, 2.1% cut-off (95% range)", fontsize=10, loc="left")
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", alpha=.25); fig.tight_layout(); fig.savefig(FIGURES / "11_wrongly_declined.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4.6))
for (t, g), c, mk in zip(cal.reset_index().groupby("tenure"), ["#0072B2", "#009E73", "#D55E00"], ["o", "s", "^"]):
    g = g[g.loans >= 50]
    ax.plot(g.predicted, g.actual, marker=mk, color=c, lw=2, ms=8, label=t)
ax.plot([.001, .5], [.001, .5], color="#777777", lw=1, ls="--", label="predicted = actual")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Predicted chance of default"); ax.set_ylabel("Actual default rate")
ax.set_xticks([.001, .01, .1, .5], ["0.1%", "1%", "10%", "50%"]); ax.set_yticks([.001, .01, .1, .5], ["0.1%", "1%", "10%", "50%"])
ax.set_title("Does a score mean the same for each group? (bands with 50+ loans)", fontsize=10, loc="left")
ax.legend(frameon=False, fontsize=9); ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(FIGURES / "11_calibration_by_tenure.png", dpi=150); plt.close(fig)
print("saved 2 figures")

saved 2 figures


## 17. What this means

The conclusions are written up in [learnings/11](../learnings/11_fairness.md), and the decision is in [ADR 0020](../docs/adr/0020-fairness-mitigation.md). In short: keep one honest score for everyone, drop the three 'borrow often' clues, and give first-time applicants a small starter loan instead of a no. Group cut-offs, reweighing and hiding tenure all lost. `src/fairness_scenarios.py` then tested 1,224 combinations and added a 24% risk cap in Kenya ([learnings/11](../learnings/11_fairness.md) §9).